# Bacpipe Tutorial
This notebook walks through the main workflows of the `bacpipe` library — from inspecting configuration and loading audio, through generating embeddings with built-in and custom models, to running the full pipeline and benchmarking classifier performance.

---
## 1. Setup & Configuration
Import necessary modules and set the working directory to the repository root.

In [ ]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os 

# replace this with the path to the directory on your machine
# os.chdir('../..')
os.chdir('/media/haupert/data/mes_projets/_packages/bacpipe.git/bacpipe')

# import the package
import bacpipe

---
## 3. Extending an Existing Model
Subclass an existing bacpipe model to modify its behaviour — for example, squaring the input before passing it through BirdNET.

In [ ]:
# Select the class Model of the desired model, here birdnet, but it could be another model in the list.
# To know all supported models, run bacpipe.supported_models.
# For instance, if one wants to use birdMAE, the code is 
# from bacpipe.model_pipelines.feature_extractors.birdmae import Model

from bacpipe.model_pipelines.feature_extractors.birdnet import Model

# Create a subclass from the class Model depending on the chosen model
class MyBirdNETModel(Model):
    def __call__(self, input):
        input = input ** 2
        return self.embeds(input, training=False)

---
## 4. Probing Pipeline
Train a linear probe on top of frozen embeddings to evaluate how well the representations capture your specific ground truth annotations.

In [ ]:
embeds = bacpipe.run_pipeline_for_single_model(
    model_name='mel',                               # name of the model to run. Supported models are in bacpipe.supported_models
    audio_dir='bacpipe/tests/test_data',                # path to directory containing audio files  
    CustomModel=MyModel
).embeddings(return_type='array')

In [ ]:
# Run this function after computing the embeddings other it is no able to find the connection between embeddings and labels
gt = bacpipe.ground_truth_by_model(
    'mel', 
    'bacpipe/tests/test_data', 
    annotations_filename='annotations.csv',
    single_label=False,
    overwrite=False
)

In [ ]:
probe, label2idx, metrics = bacpipe.probing_pipeline(
    model_name='custom_birdnet', 
    ground_truth=gt,
    embeds=embeds)

In [ ]:
embeds = bacpipe.run_pipeline_for_single_model(
    model_name='custom_birdnet',                               # name of the model to run. Supported models are in bacpipe.supported_models
    audio_dir='bacpipe/tests/test_data',                # path to directory containing audio files  
    CustomModel=MyBirdNETModel
).embeddings(return_type='array')

# Run this function after computing the embeddings other it is no able to find the connection between embeddings and labels
gt = bacpipe.ground_truth_by_model(
    'custom_birdnet', 
    'bacpipe/tests/test_data', 
    annotations_filename='annotations.csv',
    single_label=False,
    overwrite=False
)

probe, label2idx, metrics = bacpipe.probing_pipeline(
    model_name='custom_birdnet', 
    ground_truth=gt,
    embeds=embeds)

In [ ]:
embeds = bacpipe.run_pipeline_for_single_model(
    model_name='birdnet',                               # name of the model to run. Supported models are in bacpipe.supported_models
    audio_dir='bacpipe/tests/test_data',                # path to directory containing audio files  
).embeddings(return_type='array')

# embeds = bacpipe.Loader(
#     'bacpipe/tests/test_data', 
#     'birdnet2',
#     use_folder_structure=True,
#     CustomModel=MyBirdNETModel
# ).embeddings(return_type='array')

# Run this function after computing the embeddings other it is no able to find the connection between embeddings and labels
gt = bacpipe.ground_truth_by_model(
    'birdnet', 
    'bacpipe/tests/test_data', 
    annotations_filename='annotations.csv',
    single_label=False,
    overwrite=False
)


probe, label2idx, metrics = bacpipe.probing_pipeline(
    model_name='birdnet', 
    ground_truth=gt,
    embeds=embeds)


In [ ]:
print("======== display probe ==========")
display(type(probe))
print("======== display label2idx ======")
display(label2idx)
print("======== display metrics ========")
display(metrics)

---
## 5. Probe Inference
Load a previously trained linear probe and run inference to generate new predictions from your embeddings.

In [ ]:
# !BUG : threshold should be optional as it is not necessary when return_binary_presence is False

probe, label2idx = bacpipe.prepare_probe_inference(model='birdnet')

predictions = bacpipe.run_probe_inference(
    model='birdnet',
    linear_probe=probe,
    threshold=0.5,
    embeds=embeds,
    device='cuda', 
    return_binary_presence=False
)

display(predictions)


---
## 6. Clustering Pipeline
Run bacpipe's built-in clustering evaluation pipeline to group embeddings and compare the clusters against your ground truth labels.

In [ ]:
gt = bacpipe.ground_truth_by_model(
    'birdnet', 
    'bacpipe/tests/test_data', 
    annotations_filename='annotations.csv',
    single_label=False,
    overwrite=False
)

embeds = bacpipe.Loader(
    'bacpipe/tests/test_data', 
    'birdnet',
    use_folder_structure=True
).embeddings(return_type='array')

clusterings, metrics = bacpipe.clustering_pipeline('birdnet', gt, embeds)

---
## 7. Custom Clustering & Evaluation
Inject your own custom clustering algorithms (like sklearn's KMeans) and evaluate their performance using metrics like the Silhouette score.

In [ ]:
from sklearn.cluster import KMeans

# Make sure embeddings are generated
loader_obj = bacpipe.generate_embeddings(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data'
)

# Run custom clustering without ground truth
clustered_points = bacpipe.run_clustering(
    embeds, 
    {'my_clustering': KMeans(n_clusters=3)}
)

# Run custom clustering with ground truth for alignment/evaluation
clustered_points_gt = bacpipe.run_clustering(
    embeds, 
    {'my_clustering': KMeans(n_clusters=3)}, 
    ground_truth=gt['label:species']
)

# Evaluate embedding separation using Silhouette score
metrics = bacpipe.eval_with_silhouette(
    embeds, 
    ground_truth=gt['label:species']
)